In [ ]:
from __future__ import annotations

import io
import os
from pathlib import Path

import pandas as pd


def find_workspace_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "train_parallel.py").exists() and (candidate / "functions").exists():
            return candidate
    raise FileNotFoundError("Could not locate the GeoPAS workspace root from the current working directory.")


WORKSPACE_ROOT = find_workspace_root(Path.cwd().resolve())
PROJECT_ROOT = Path(
    os.environ.get("PROJECT_ROOT", os.environ.get("GEOPAS_PROJECT_ROOT", str(WORKSPACE_ROOT.parent)))
).resolve()
DEFAULT_RESULT_ROOT = PROJECT_ROOT / "results" / "bbob_by_deepela" / "results"

# --- Config ---
RESULT_ROOT = Path(os.environ.get("RESULTS_ROOT", str(DEFAULT_RESULT_ROOT))).resolve()
GRID_ROOT = RESULT_ROOT / "bbob"
SPLITS = ["LPO", "LIO", "RANDOM"]
FILENAME_GLOB = "res_tailpenalty_*.csv"

# If you want shorter section names, set this to True.
SHORTEN_SECTION_NAMES = False

def _extract_table_block(lines: list[str], stat: str, substat: str) -> pd.DataFrame:
    """Extract a rectangular CSV table that starts at:
        <stat>
        <substat>
        <header, comma-separated>
        <rows...>
        <blank line>
    Returns a DataFrame indexed by the first column (e.g. 'Problem Group').
    """
    start = None
    for i in range(len(lines) - 1):
        if lines[i].strip() == stat and lines[i + 1].strip() == substat:
            start = i + 2
            break
    if start is None:
        raise ValueError(f"Could not find section '{stat} -> {substat}'")

    header = lines[start].strip()
    if not header:
        raise ValueError(f"Empty header for section '{stat} -> {substat}'")

    data_lines: list[str] = []
    for j in range(start + 1, len(lines)):
        line = lines[j].strip()
        if line == "":
            break
        data_lines.append(lines[j])

    block = "\n".join([header] + data_lines)
    df = pd.read_csv(io.StringIO(block))
    if df.shape[1] < 2:
        raise ValueError(
            f"Parsed table for '{stat} -> {substat}' looks wrong: columns={df.columns.tolist()}"
        )

    df = df.set_index(df.columns[0])
    return df

def read_mean_median_as_tables(
    csv_path: Path,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Return the AS tables for mean, median, p90, and their log-domain counterparts."""
    text = csv_path.read_text()
    lines = text.splitlines()
    mean_as = _extract_table_block(lines, "Mean", "AS")
    median_as = _extract_table_block(lines, "Median", "AS")
    p90_as = _extract_table_block(lines, "P90", "AS")
    log_mean_as = _extract_table_block(lines, "Log_Mean", "AS")
    log_median_as = _extract_table_block(lines, "Log_Median", "AS")
    log_p90_as = _extract_table_block(lines, "Log_P90", "AS")
    return mean_as, median_as, p90_as, log_mean_as, log_median_as, log_p90_as

def _maybe_shorten(section: str) -> str:
    if not SHORTEN_SECTION_NAMES:
        return section
    return section.replace("tail1_", "").replace("tail0_", "").replace("dual1_", "dual=")

def _iter_split_csv_paths(split: str) -> list[Path]:
    pattern = f"*/**/{split}/{FILENAME_GLOB}"
    csv_paths = sorted(GRID_ROOT.glob(pattern))
    if not csv_paths:
        raise FileNotFoundError(f"No CSVs found for split={split} under {GRID_ROOT} via {pattern}")
    return csv_paths

def _build_section_name(csv_path: Path, split: str) -> str:
    parts = csv_path.relative_to(GRID_ROOT).parts
    if not parts:
        raise ValueError(f"Could not derive a relative path under {GRID_ROOT} for {csv_path}")

    parameter_set = _maybe_shorten(parts[0])
    split_indices = [index for index, part in enumerate(parts[:-1]) if part == split]
    if not split_indices:
        raise ValueError(f"Could not find split directory '{split}' inside {csv_path}")

    split_index = split_indices[-1]
    seed = None
    if split_index >= 1 and parts[split_index - 1].lower().startswith("seed"):
        seed = parts[split_index - 1]

    section_parts = [parameter_set]
    if seed is not None:
        section_parts.append(seed)
    section_parts.append(csv_path.stem)
    return " -- ".join(section_parts)

def build_sectioned_csv_for_split(split: str) -> Path:
    csv_paths = _iter_split_csv_paths(split)

    out_path = RESULT_ROOT / f"AS_mean_median_p90__{split}__ALL_RUNS.csv"
    bad: list[tuple[Path, str]] = []
    wrote = 0

    with out_path.open("w", encoding="utf-8", newline="\n") as f:
        for csv_path in csv_paths:
            section = _build_section_name(csv_path, split)

            try:
                mean_as, median_as, p90_as, log_mean_as, log_median_as, log_p90_as = read_mean_median_as_tables(csv_path)
            except Exception as e:
                bad.append((csv_path, str(e)))
                continue

            f.write(f"{section}\n")
            f.write("Mean\n")
            mean_as.to_csv(f)
            f.write("Median\n")
            median_as.to_csv(f)
            f.write("P90\n")
            p90_as.to_csv(f)
            f.write("Log_Mean\n")
            log_mean_as.to_csv(f)
            f.write("Log_Median\n")
            log_median_as.to_csv(f)
            f.write("Log_P90\n")
            log_p90_as.to_csv(f)
            f.write("\n")
            wrote += 1

    if bad:
        print(f"\n[WARN] Failed to parse {len(bad)} file(s) for split={split}:")
        for p, msg in bad[:20]:
            print(f"  - {p}: {msg}")
        if len(bad) > 20:
            print(f"  ... and {len(bad) - 20} more")

    print(f"Wrote {out_path} with {wrote} parameter set(s) (each includes Mean+Median+P90 tables).")
    return out_path

def build_all_available_splits(splits: list[str]) -> dict[str, Path]:
    outs: dict[str, Path] = {}
    missing: list[str] = []

    for split in splits:
        try:
            outs[split] = build_sectioned_csv_for_split(split)
        except FileNotFoundError as exc:
            print(f"[WARN] {exc}")
            missing.append(split)

    if not outs:
        raise FileNotFoundError(f"No result CSVs found under {GRID_ROOT} for any of {splits}")

    if missing:
        print(f"[WARN] Skipped unavailable splits: {', '.join(missing)}")

    return outs

# --- Run for all available splits ---
outs = build_all_available_splits(SPLITS)
outs

Wrote /data1/home/jw1017/AS_BBO_REBUILT/results/bbob_by_deepela/results/AS_mean_median_p90__LPO__ALL_RUNS.csv with 32 parameter set(s) (each includes Mean+Median+P90 tables).
Wrote /data1/home/jw1017/AS_BBO_REBUILT/results/bbob_by_deepela/results/AS_mean_median_p90__LIO__ALL_RUNS.csv with 32 parameter set(s) (each includes Mean+Median+P90 tables).
Wrote /data1/home/jw1017/AS_BBO_REBUILT/results/bbob_by_deepela/results/AS_mean_median_p90__RANDOM__ALL_RUNS.csv with 32 parameter set(s) (each includes Mean+Median+P90 tables).


{'LPO': PosixPath('/data1/home/jw1017/AS_BBO_REBUILT/results/bbob_by_deepela/results/AS_mean_median_p90__LPO__ALL_RUNS.csv'),
 'LIO': PosixPath('/data1/home/jw1017/AS_BBO_REBUILT/results/bbob_by_deepela/results/AS_mean_median_p90__LIO__ALL_RUNS.csv'),
 'RANDOM': PosixPath('/data1/home/jw1017/AS_BBO_REBUILT/results/bbob_by_deepela/results/AS_mean_median_p90__RANDOM__ALL_RUNS.csv')}

In [ ]:
from __future__ import annotations

import os
from pathlib import Path


def find_workspace_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "train_parallel.py").exists() and (candidate / "functions").exists():
            return candidate
    raise FileNotFoundError("Could not locate the GeoPAS workspace root from the current working directory.")


WORKSPACE_ROOT = find_workspace_root(Path.cwd().resolve())
PROJECT_ROOT = Path(
    os.environ.get("PROJECT_ROOT", os.environ.get("GEOPAS_PROJECT_ROOT", str(WORKSPACE_ROOT.parent)))
).resolve()
DEFAULT_RESULT_ROOT = PROJECT_ROOT / "results" / "bbob_by_deepela" / "results"
RESULT_ROOT = Path(os.environ.get("RESULTS_ROOT", str(DEFAULT_RESULT_ROOT))).resolve()

# Inputs (edit if you want different paths/order)
FILES = {
    "LPO": RESULT_ROOT / "AS_mean_median_p90__LPO__ALL_RUNS.csv",
    "LIO": RESULT_ROOT / "AS_mean_median_p90__LIO__ALL_RUNS.csv",
    "RANDOM": RESULT_ROOT / "AS_mean_median_p90__RANDOM__ALL_RUNS.csv",
}
OUT = RESULT_ROOT / "AS_mean_median_p90__MERGED__ALL_RUNS.csv"

def parse_sectioned_csv(path: Path) -> dict[str, dict[str, list[str]]]:
    """Parse: section -> {'Mean': [...], 'Median': [...], 'P90': [...]} preserving raw CSV text."""
    lines = path.read_text(encoding="utf-8").splitlines()
    i, n = 0, len(lines)
    out: dict[str, dict[str, list[str]]] = {}

    def skip_blanks(j: int) -> int:
        while j < n and lines[j].strip() == "":
            j += 1
        return j

    while True:
        i = skip_blanks(i)
        if i >= n:
            break
        section = lines[i].strip()
        i += 1

        i = skip_blanks(i)
        if i >= n or lines[i].strip() != "Mean":
            raise ValueError(f"{path}: expected 'Mean' after section '{section}'")
        i += 1
        mean: list[str] = []
        while i < n and lines[i].strip() != "Median":
            mean.append(lines[i])
            i += 1

        if i >= n or lines[i].strip() != "Median":
            raise ValueError(f"{path}: missing 'Median' for section '{section}'")
        i += 1
        median: list[str] = []
        while i < n and lines[i].strip() != "P90":
            median.append(lines[i])
            i += 1

        if i >= n or lines[i].strip() != "P90":
            raise ValueError(f"{path}: missing 'P90' for section '{section}'")
        i += 1
        p90: list[str] = []
        while i < n and lines[i].strip() != "":
            p90.append(lines[i])
            i += 1

        out[section] = {"Mean": mean, "Median": median, "P90": p90}
    return out

def get_last_metric_value(rows: list[str]) -> float:
    if not rows:
        return float("inf")
    return float(rows[-1].rsplit(",", 1)[-1])

available_files = {split: path for split, path in FILES.items() if path.exists()}
missing_files = [split for split, path in FILES.items() if not path.exists()]

if not available_files:
    raise FileNotFoundError(
        "No per-split CSVs found. Run the previous cell first, or update FILES to point to existing outputs."
    )

if missing_files:
    print(f"[WARN] Missing split outputs: {', '.join(missing_files)}")

parsed = {split: parse_sectioned_csv(path) for split, path in available_files.items()}
order = list(parsed[next(iter(parsed))].keys())  # keep section coverage from the first available file

if "LPO" in parsed:
    original_index = {section: index for index, section in enumerate(order)}
    order.sort(
        key=lambda section: (
            section not in parsed["LPO"],
            get_last_metric_value(parsed["LPO"][section]["Mean"]) if section in parsed["LPO"] else float("inf"),
            original_index[section],
        )
    )

missing = []
for section in order:
    for split in available_files:
        if section not in parsed[split]:
            missing.append((section, split))
if missing:
    print("[WARN] Missing sections:")
    for section, split in missing[:20]:
        print(f"  - {section} missing in {split}")
    if len(missing) > 20:
        print(f"  ... and {len(missing) - 20} more")

with OUT.open("w", encoding="utf-8", newline="\n") as f:
    for section in order:
        violate = False
        for split in available_files:

            if split == "LPO":
                threshold_mean = 40
                threshold_median = 2
                threshold_p90 = 10
            else:
                threshold_mean = 30
                threshold_median = 2
                threshold_p90 = 8

            if section not in parsed[split]:
                continue

            violate = float(parsed[split][section]["Mean"][-1].split(',')[-1]) >= threshold_mean or \
                    float(parsed[split][section]["Median"][-1].split(',')[-1]) >= threshold_median or \
                    float(parsed[split][section]["P90"][-1].split(',')[-1]) >= threshold_p90 or \
                    violate

        if violate:
            continue

        f.write(section + "\n")
        for split in available_files:
            if section not in parsed[split]:
                continue

            f.write(split + "\n")
            f.write("Mean\n")
            f.write("\n".join(parsed[split][section]["Mean"]) + "\n")
            f.write("Median\n")
            f.write("\n".join(parsed[split][section]["Median"]) + "\n")
            f.write("P90\n")
            f.write("\n".join(parsed[split][section]["P90"]) + "\n")
        f.write("\n")

print(f"Wrote {OUT}")

Wrote /data1/home/jw1017/AS_BBO_REBUILT/results/bbob_by_deepela/results/AS_mean_median_p90__MERGED__ALL_RUNS.csv
